# PhishRadar: LLM prompting

Classify each email as phishing/scam (1) or legitimate (0) by prompting `claude-haiku-5-5`. Nothing is trained. Only the prompt changes. The logic lives in `src/llm_classifier.py`; this notebook drives it.

**Rules (same as Approaches A and B)**
- Develop and tweak prompts on `val.csv` only. Test files are loaded only in Step 4, after the prompts are frozen.
- Freeze the prompts, then run each frozen prompt on the test samples once.
- Score TF-IDF and DistilBERT on the same sampled rows so the comparison is fair.

**Workflow**

| Step | What happens | Costs money? |
|---|---|---|
| 0 | Leave `BACKEND = "mock"` and run every cell top to bottom | No |
| 1 | Dry run on 20 val emails: check parsing and the projected cost | A few cents |
| 2 | Iterate on the prompt files using the val dev sample | Yes, small |
| 3 | Freeze the prompts | No |
| 4 | Run the frozen prompts on the test samples | Yes, small |
| 5 | Metrics with bootstrap CIs, compared with TF-IDF and DistilBERT on the same rows | No |

**Mock backend:** a fake model that uses a keyword rule. It costs nothing and lets you test sampling, prompts, the cache, failure counting and output files. Its predictions mean nothing. Mock results are written under `results/llm_cache/` (git-ignored), so they can never mix with real results.

**Setup.** Locally (recommended, no GPU needed): from the repo root, `python -m venv venv`, `venv/bin/pip install -r requirements.txt`, put the split CSVs in `data/splits/` (see `data/data_README.md`; `results/samples/splits_checksums.json` lets you confirm your copy matches), and open this notebook from the repo root. On Colab: keep a copy of the repo on Google Drive (set `REPO_ROOT` below) and store the key under Secrets as `ANTHROPIC_API_KEY`.

**Before committing:** clear all outputs (some cells print email text) and never commit `.env`.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic", "python-dotenv"], check=True)

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

pd.set_option("display.max_colwidth", 120)

## 1. Configuration

The settings you are most likely to change are `BACKEND`, `SETTINGS`, `SAMPLE_SIZES` and `MAX_WORKERS`. Write the values in `SETTINGS` and `SAMPLE_SIZES` into the report.

In [ ]:
# ---- Backend ----
# "mock"      fake model, no API calls, $0. Use it to build and test the pipeline.
# "anthropic" real Claude API calls. Needs ANTHROPIC_API_KEY in .env and credits.
BACKEND = "mock"

# ---- Paths: run this notebook from the repo root ----
REPO_ROOT = Path.cwd()
if IN_COLAB:
    # On Colab, keep the repo (and the cache) on Drive so nothing is lost when the runtime resets.
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/drive/MyDrive/PhishRadar/PhishRadar-Project")   # change to your copy
sys.path.insert(0, str(REPO_ROOT / "src"))
import llm_classifier as llm

SPLITS_DIR  = REPO_ROOT / "data" / "splits"
PROMPTS_DIR = REPO_ROOT / "prompts"
RESULTS_DIR = REPO_ROOT / "results"
SAMPLES_DIR = RESULTS_DIR / "samples"       # committed: row_id lists only, no text
PRED_DIR    = RESULTS_DIR / "predictions"   # committed: row_id, y_true, y_pred, score
CACHE_DIR   = RESULTS_DIR / "llm_cache"     # git-ignored: contains email text
for d in (SAMPLES_DIR, PRED_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- API key ----
load_dotenv(REPO_ROOT / ".env")
if IN_COLAB and not os.environ.get("ANTHROPIC_API_KEY"):
    try:   # Colab: store the key under the key icon (Secrets) as ANTHROPIC_API_KEY
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        pass

# ---- Request and text settings (state these in the report) ----
# Haiku 5.5 takes no temperature/top_p/top_k, so the same request can get different answers.
# The cache is what makes results reproducible: an identical request always returns the stored answer.
SETTINGS = llm.Settings(
    model=os.environ.get("LLM_MODEL", "claude-haiku-5-5"),
    max_tokens=512,           # room for the tool call plus a short reason
    max_chars=2000,           # every email is cut to this many characters, for every prompt
    fewshot_max_chars=600,    # few-shot examples are cut shorter to keep the prompt small
    normalize_spacing=True,   # same normalize_spacing as the DistilBERT notebook
)

# ---- Sampling ----
SEED = 42
DEV_N = 300               # val emails used for prompt development
DRY_RUN_N = 20            # val emails for the first dry run
TEST_FILES = {
    "insource":       "test_insource.csv",
    "ceas":           "test_ceas.csv",
    "nazario_paired": "test_nazario_paired.csv",   # Nazario recall comes from its phishing half
    "nigerian":       "test_nigerian.csv",
}
SAMPLE_SIZES = {name: 1000 for name in TEST_FILES}

# ---- Runtime ----
MAX_WORKERS = 4           # parallel API calls; the SDK backs off on rate limits (429)

print(f"Backend: {BACKEND} | Model: {SETTINGS.model} | Repo: {REPO_ROOT}")

## 2. Load train and val

`row_id` is the row position in the split file, the same as in the TF-IDF (`row_id`) and DistilBERT (`row`) prediction files. This was checked by joining each committed prediction file to its split and comparing labels on every row. Step 5 repeats the check before every comparison.

The val dev sample is drawn once with `SEED`, stratified by label, and saved to `results/samples/val_dev_rowids.csv`. After that the saved file is always loaded, never redrawn. Commit the files in `results/samples/` so teammates can score their models on exactly the same rows.

In [ ]:
train_df = llm.load_split(SPLITS_DIR / "train.csv")
val_df = llm.load_split(SPLITS_DIR / "val.csv")
display(pd.DataFrame([{"split": n, "rows": len(d), "phishing_share": round(d["label"].mean(), 3)}
                      for n, d in [("train", train_df), ("val", val_df)]]))

dev_df = llm.fixed_sample(val_df, "val_dev", DEV_N, SAMPLES_DIR, SEED)

## 3. Prompt variants

The five variants form a ladder. Each one changes a single thing, so any difference in results can be traced to that change (research question 4).

| Variant | What it adds | Question it answers |
|---|---|---|
| `zs_plain` | Nothing: a bare question | How well does the model do with no guidance? |
| `zs_definition` | The project's broad definition (spam + scams + credential phishing = 1) | Does telling it what "phishing" means here help? |
| `zs_cues` | `zs_definition` + a red-flag checklist | Do explicit hints help, or make it flag too much? |
| `zs_reasoned` | `zs_cues` + a one-sentence reason written before the label | Does a short "think first" step help? |
| `fs_balanced` | `zs_definition` + 6 hand-picked examples from `train.csv` | Do examples help? Compare with `zs_definition`. |

**How the answer comes back.** Every variant forces the model to answer by calling a `classify_email` tool with fixed fields (`label`, `phishing_probability`, and `reason` for `zs_reasoned`). The output always has the right shape, and a forced tool call skips the model's thinking step, so calls are cheaper and more consistent.

**File format.** Each prompt is a text file in `prompts/` with a `=== SYSTEM ===` section and a `=== USER ===` section. `{email}` in the user section is replaced by the email. `{examples}` in the system section is replaced by the few-shot examples. Lines above `=== SYSTEM ===` are comments. The files are part of the experiment: edit them there, and add a variant by creating a file and adding an entry to `VARIANTS`.

In [ ]:
VARIANTS = dict(llm.VARIANTS)
# RQ4, example order: same examples, different order. Uncomment to test ordering sensitivity.
# VARIANTS["fs_balanced_order1"] = {"prompt": "fs_balanced.txt", "reason": False, "fewshot": True, "order_seed": 1}
# VARIANTS["fs_balanced_order2"] = {"prompt": "fs_balanced.txt", "reason": False, "fewshot": True, "order_seed": 2}

missing = [v["prompt"] for v in VARIANTS.values() if not (PROMPTS_DIR / v["prompt"]).exists()]
if missing:
    raise FileNotFoundError(f"Missing prompt files in {PROMPTS_DIR}: {missing}")
display(pd.DataFrame(VARIANTS).T)

### Few-shot examples

Examples come from `train.csv` only. They were picked by hand from readable candidates (see `fewshot_candidates` in `src/llm_classifier.py`): one per category of the project's definition. For phishing: credential-style, scam/fraud, bulk ad. For legitimate: work, personal, mailing list or newsletter. The picks and the criteria are saved in `prompts/fewshot_examples.json` (row_ids only, no text).

Credential phishing (Nazario) and advance-fee scams (Nigerian) exist as sources only in the test sets, so the phishing examples are the closest matches found inside `train.csv`. Taking examples from a test source would leak it.

The credential example (row 21659) and the scam example (row 25535) resemble the Nazario and Nigerian test categories but come from `train.csv` only.

If the JSON file is missing, the cell below prints candidates and stops.

In [ ]:
FEWSHOT_FILE = PROMPTS_DIR / "fewshot_examples.json"

if not FEWSHOT_FILE.exists():
    cands = llm.fewshot_candidates(train_df)
    for _, r in cands.iterrows():
        print(f"\n[{r['category']}] row_id={r['row_id']} source={r['source']} label={r['label']}\n{r['text'][:300]}")
    raise RuntimeError(f"Pick the few-shot examples and record them in {FEWSHOT_FILE}.")

FEWSHOT = llm.load_fewshot(FEWSHOT_FILE, train_df)
meta = json.loads(FEWSHOT_FILE.read_text())
print("Criteria:\n- " + "\n- ".join(meta["criteria"]))
for e, (_, r) in zip(meta["examples"], FEWSHOT.iterrows()):
    tag = "PHISHING" if r["label"] == 1 else "LEGIT"
    print(f"\n[{tag} / {e.get('category')}] row_id={r['row_id']} source={r['source']}\n{r['text'][:250]}...")

## 4. Backend, cache and one rendered request

- **`AnthropicBackend`** makes the real call with a forced tool call. A refusal (`stop_reason == "refusal"`) and a missing or invalid tool call are recorded as failures, never dropped. The system prompt is marked for prompt caching, which lowers the price of repeated prompts without changing answers.
- **`MockBackend`** fakes answers with a keyword rule. About 1% of its answers are fake refusals and 2% are malformed, so the failure handling gets tested too.
- **Cache:** every completed answer (including refusals) is stored in `results/llm_cache/cache_<backend>.jsonl`, keyed on a hash of the full request: model, exact prompt text, email text and tool schema. Editing a prompt can never return a stale answer, and rerunning an unchanged request costs nothing. Network or rate-limit errors are not cached, so a rerun retries them. Errors that would hit every row (bad request, bad key) stop the run.
- **Failed rows** get `y_pred = -1`. Metrics count them as wrong, and `n_failed` reports how many there were.

Every email goes through the same steps for every variant: DistilBERT's `normalize_spacing`, cut to `max_chars`, and neutralize any `<email>`, `<example>` or `<label>` tags inside it so an email can't close our tags early.

In [ ]:
backend = llm.AnthropicBackend(SETTINGS) if BACKEND == "anthropic" else llm.MockBackend()
cache = llm.ResponseCache(CACHE_DIR / f"cache_{backend.name}.jsonl")
clf = llm.LLMClassifier(SETTINGS, PROMPTS_DIR, backend, cache, VARIANTS, FEWSHOT, MAX_WORKERS)
print(f"Using the {backend.name} backend | cache: {len(cache)} stored answers"
      + (f", {cache.skipped_lines} unreadable lines skipped" if cache.skipped_lines else ""))

PREVIEW_VARIANT = "fs_balanced"
_s, _u, _t = clf.build_variant(PREVIEW_VARIANT)
print("=== SYSTEM ===\n" + _s)
print("\n=== USER ===\n" + clf.render_user(_u, dev_df["text"].iloc[0])[:800])
print("\n=== TOOL ===\n" + json.dumps(_t, indent=2))

## Step 1: Dry run (20 val emails)

Runs every variant on 20 val emails. Check that nothing fails to parse, look at the tokens per call, and read the projected cost of the whole plan before spending more.

With the mock backend, input tokens are estimates (characters / 4, plus 30% for the newer tokenizer, plus the 406 tokens the API adds for a forced tool call) and output tokens are a guess. The projection charges every input token at the full price, so it is an upper bound: prompt caching only lowers it. For exact input counts, the free token count cell after this one runs automatically when an API key is available.

In [ ]:
RUN_DRY_RUN = True
DEV_ROUNDS_ESTIMATE = 5    # your guess at how many times you'll rerun val while iterating on prompts
OUT_TOKENS_GUESS = {False: 50, True: 90}   # mock only: output tokens per call without / with a reason

CALLS = {"dry run": DRY_RUN_N, "prompt development": DEV_N * DEV_ROUNDS_ESTIMATE,
         "test step": sum(SAMPLE_SIZES.values())}

if RUN_DRY_RUN:
    dry_df = dev_df.sample(n=min(DRY_RUN_N, len(dev_df)), random_state=SEED)
    rows = []
    for v in VARIANTS:
        out = clf.run_variant(v, dry_df, desc=f"dry run {v}")
        avg_out = (out["output_tokens"].mean() if backend.name == "anthropic"
                   else OUT_TOKENS_GUESS[VARIANTS[v].get("reason", False)])
        rows.append({"variant": v, "ok": int((out["status"] == "ok").sum()),
                     "failed": int((out["status"] != "ok").sum()),
                     "avg_in_tokens": out["input_tokens"].mean() + out["cache_read_input_tokens"].mean()
                                      + out["cache_creation_input_tokens"].mean(),
                     "avg_out_tokens": avg_out})
    dry = pd.DataFrame(rows)
    for stage, n in CALLS.items():
        dry[f"cost_{stage.replace(' ', '_')}"] = [llm.project_cost(i, o, n)
                                                 for i, o in zip(dry["avg_in_tokens"], dry["avg_out_tokens"])]
    display(dry.round(4))
    total = dry.filter(like="cost_").to_numpy().sum()
    print(f"Calls per variant: {CALLS} -> {sum(CALLS.values()):,} x {len(VARIANTS)} variants")
    print(f"Projected cost of the test step: ${dry['cost_test_step'].sum():.2f}")
    print(f"Projected cost of the whole plan (dry run + {DEV_ROUNDS_ESTIMATE} dev rounds + test): ${total:.2f}")
    if backend.name == "mock":
        print("(Mock backend: token counts are estimates. Upper bound: ignores prompt-caching savings.)")

### Free token count (no credits used)

The token counting endpoint is free and only needs an API key, so this runs whenever one is set, even while `BACKEND` is still `"mock"`. It counts exact input tokens for 10 val emails per variant.

In [ ]:
RUN_FREE_TOKEN_COUNT = bool(os.environ.get("ANTHROPIC_API_KEY"))

if RUN_FREE_TOKEN_COUNT:
    counter = llm.AnthropicBackend(SETTINGS)
    texts = dev_df["text"].sample(n=min(10, len(dev_df)), random_state=SEED)
    rows = []
    for v in VARIANTS:
        system, user_template, tool = clf.build_variant(v)
        counts = [counter.count_tokens(system, clf.render_user(user_template, t), tool) for t in texts]
        rows.append({"variant": v, "avg_input_tokens": float(np.mean(counts))})
    tc = pd.DataFrame(rows)
    tc["test_step_input_cost_usd"] = [llm.project_cost(i, 0, CALLS["test step"]) for i in tc["avg_input_tokens"]]
    display(tc.round(4))
    print(f"Projected test-step input cost: ${tc['test_step_input_cost_usd'].sum():.2f} "
          "(output adds roughly 20-90 tokens per call)")
else:
    print("No ANTHROPIC_API_KEY: skipped. The dry run above shows the estimate.")

## Step 2: Develop prompts on val

Run every variant on the val dev sample, look at the metrics and the mistakes, edit the files in `prompts/`, and run again. Unchanged variants come straight from the cache, so only edited prompts cost anything.

Things worth trying while you iterate:
- Rewording the definition or cues when one kind of mistake dominates (for example, newsletters flagged as spam).
- Uncommenting the `fs_balanced_order*` variants to see how much example order matters.
- Writing a variant of your own: new file in `prompts/`, new entry in `VARIANTS`.

Record what you changed and why. It becomes the prompt-development section of the report.

In [ ]:
RUN_DEV = True

if RUN_DEV:
    dev_results = {v: clf.run_variant(v, dev_df, desc=f"val {v}") for v in VARIANTS}
    dev_table = pd.DataFrame([{"variant": v, **llm.compute_metrics(o["y_true"], o["y_pred"])}
                              for v, o in dev_results.items()])
    display(dev_table[["variant", "n", "n_failed", "precision", "recall", "f1", "fpr", "accuracy"]].round(3))

In [ ]:
INSPECT_VARIANT = "zs_cues"
N_SHOW = 5

if RUN_DEV:
    o = dev_results[INSPECT_VARIANT].merge(dev_df[["row_id", "text", "source"]], on="row_id")
    groups = {
        "False alarms (legitimate flagged as phishing)": o[(o["y_true"] == 0) & (o["y_pred"] == 1)],
        "Missed phishing": o[(o["y_true"] == 1) & (o["y_pred"] == 0)],
        "Failed calls": o[o["y_pred"] == -1],
    }
    for title, g in groups.items():
        print(f"\n===== {title}: {len(g)} =====")
        for _, r in g.head(N_SHOW).iterrows():
            print(f"\nrow_id={r['row_id']} source={r['source']} status={r['status']} "
                  f"prob={r['prob']} reason={r['reason']}")
            print(r["text"][:400].replace("\n", " "))

## Step 3: Freeze the prompts

When you are happy with the val results, set `DO_FREEZE = True`, run the cell once, then set it back to `False`.

This saves a fingerprint of every variant to `results/llm_prompt_freeze.json`: model, settings, the full system prompt with examples filled in, the user template and the tool schema. With the real backend, Step 4 refuses to run if any variant changed after the freeze.

Then commit and tag:

```bash
git add prompts/ results/llm_prompt_freeze.json results/samples/
git commit -m "Freeze LLM prompts v1"
git tag prompts-v1
```

In [ ]:
FREEZE_FILE = RESULTS_DIR / "llm_prompt_freeze.json"

DO_FREEZE = False
if DO_FREEZE:
    llm.freeze_prompts(clf, FREEZE_FILE, llm.git_commit(REPO_ROOT))

## Step 4: Run the test samples (once)

This is the first cell that reads the test files. Each test sample is drawn once (stratified by label, `SEED`) and its row_ids are saved to `results/samples/<name>_rowids.csv`, like the val dev sample.

With the real backend this first checks the freeze, then writes `results/predictions/llm_<variant>_<testset>.csv` with `row_id, y_true, y_pred, score` (the shared format) and appends a run record to `results/llm_runs.json`. Per-row details, including the model's reasons, go to `results/llm_cache/details/` because a reason can quote the email.

With the mock backend, everything goes to `results/llm_cache/mock_predictions/` instead, and no freeze is needed. `RUN_TEST` is on by default only for the mock backend: a real test run is a deliberate step.

Rerunning is safe: identical requests come from the cache, so the files come out the same.

In [ ]:
RUN_TEST = BACKEND == "mock"

def load_test_samples() -> dict:
    return {name: llm.fixed_sample(llm.load_split(SPLITS_DIR / f), name, SAMPLE_SIZES[name], SAMPLES_DIR, SEED)
            for name, f in TEST_FILES.items()}

if RUN_TEST:
    if backend.name == "anthropic":
        llm.check_frozen(clf, FREEZE_FILE)
        out_dir, detail_dir, log_file = PRED_DIR, CACHE_DIR / "details", RESULTS_DIR / "llm_runs.json"
    else:
        out_dir = CACHE_DIR / "mock_predictions"        # fake results never touch results/predictions
        detail_dir, log_file = out_dir / "details", out_dir / "llm_runs_mock.json"
    out_dir.mkdir(parents=True, exist_ok=True)
    detail_dir.mkdir(parents=True, exist_ok=True)
    test_samples = load_test_samples()

    run = {"started_at": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
           "backend": backend.name, "settings": dict(vars(SETTINGS)),
           "git_commit": llm.git_commit(REPO_ROOT),
           "variants": {v: clf.fingerprint(v) for v in VARIANTS}, "files": []}
    for v in VARIANTS:
        for ts, sample in test_samples.items():
            out = clf.run_variant(v, sample, desc=f"test {v} / {ts}")
            name = f"llm_{v}_{ts}"
            out[["row_id", "y_true", "y_pred", "score"]].to_csv(out_dir / f"{name}.csv", index=False)
            out.to_csv(detail_dir / f"{name}_details.csv", index=False)
            run["files"].append({"file": f"{name}.csv", "rows": len(out),
                                 "statuses": out["status"].value_counts().to_dict(),
                                 "n_failed": int((out["y_pred"] == -1).sum()),
                                 "cost_usd_new_calls": round(llm.cost_usd(out[~out["cache_hit"]]), 4)})
    run["finished_at"] = pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds")
    log = json.loads(log_file.read_text()) if log_file.exists() else []
    log.append(run)
    log_file.write_text(json.dumps(log, indent=2))
    print(f"Wrote {len(run['files'])} prediction files to {out_dir}")

## Step 5: Results with 95% bootstrap CIs

Reads the LLM prediction files from disk (so it works after a restart) and the committed TF-IDF and DistilBERT predictions, **filtered to the same sampled rows**. Before comparing, it checks that labels agree on every `row_id`. If they don't, `row_id` means something different in that file, and the comparison is skipped with a warning.

`nazario (phishing half)` is recall on the Nazario emails inside `nazario_paired`, which answers RQ3. Precision and F1 are blank for it, as in the other notebooks, because it has no legitimate emails.

In [ ]:
# What the other notebooks write (checked against the committed files):
TFIDF_PATTERN = "tfidf_{test_set}.csv"                      # in results/predictions/, from PhishRadar_tfidf_baseline
TFIDF_SET_NAMES = {ts: f"test_{ts}" for ts in TEST_FILES}  # our test-set name -> name in the TF-IDF file name
BERT_FILE = RESULTS_DIR / "bert_predictions.csv"           # from PhishRadar_DistilBERT: test_set,row,label,...,pred
BERT_SET_NAMES = {ts: f"test_{ts}" for ts in TEST_FILES}   # our test-set name -> value in its test_set column

def load_tfidf(ts):
    path = PRED_DIR / TFIDF_PATTERN.format(test_set=TFIDF_SET_NAMES[ts])
    return pd.read_csv(path)[["row_id", "y_true", "y_pred"]] if path.exists() else None

def load_bert(ts):
    if not BERT_FILE.exists():
        return None
    df = pd.read_csv(BERT_FILE)
    df = df[df["test_set"] == BERT_SET_NAMES[ts]]
    return None if df.empty else df.rename(columns={"row": "row_id", "label": "y_true", "pred": "y_pred"})[
        ["row_id", "y_true", "y_pred"]]

OTHER_MODELS = {"tfidf": load_tfidf, "distilbert": load_bert}

llm_dir = PRED_DIR if backend.name == "anthropic" else CACHE_DIR / "mock_predictions"
llm_files = {(v, ts): llm_dir / f"llm_{v}_{ts}.csv" for v in VARIANTS for ts in TEST_FILES}
preds, rows = {}, []
if not any(p.exists() for p in llm_files.values()):
    print("No LLM prediction files yet. Run Step 4 first.")
else:
    if "test_samples" not in globals():
        test_samples = load_test_samples()
    for ts, sample in test_samples.items():
        for v in VARIANTS:
            if llm_files[(v, ts)].exists():
                preds[(f"llm_{v}", ts)] = pd.read_csv(llm_files[(v, ts)])[["row_id", "y_true", "y_pred"]]
        for model, loader in OTHER_MODELS.items():
            aligned = llm.align_predictions(sample, loader(ts), model, ts)
            if aligned is not None:
                preds[(model, ts)] = aligned

    for (model, ts), p in preds.items():
        rows += llm.result_rows(model, ts, p["y_true"], p["y_pred"])
    results = pd.DataFrame(rows)
    show = results[["test_set", "model", "n", "n_failed"]].copy()
    for k in ("precision", "recall", "f1", "fpr"):
        show[k] = [f"{m:.3f} [{lo:.3f}, {hi:.3f}]" if np.isfinite(m) else "n/a"
                   for m, lo, hi in zip(results[k], results[f"{k}_lo"], results[f"{k}_hi"])]
    display(show.sort_values(["test_set", "model"]).reset_index(drop=True))
    if backend.name == "anthropic":
        results.to_csv(RESULTS_DIR / "llm_results.csv", index=False)
        print("Saved results/llm_results.csv")

In [ ]:
# Is one model really better than another on a test set? Same resampled rows for both.
COMPARE = ("llm_zs_cues", "tfidf", "ceas", "f1")    # model A, model B, test set, metric

a_name, b_name, ts, metric = COMPARE
if (a_name, ts) in preds and (b_name, ts) in preds:
    m = preds[(a_name, ts)].merge(preds[(b_name, ts)], on=["row_id", "y_true"], suffixes=("_a", "_b"))
    point, lo, hi = llm.paired_bootstrap_diff(m["y_true"], m["y_pred_a"], m["y_pred_b"], metric)
    verdict = "a real difference" if (lo > 0 or hi < 0) else "not distinguishable at this sample size"
    print(f"{metric}: {a_name} minus {b_name} on {ts} = {point:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}] -> {verdict}")
else:
    print(f"Need predictions for both {a_name} and {b_name} on {ts}. Available: {sorted(preds)}")

## Stretch: classify any list of emails

For the adversarial set (RQ5): pass any list of email texts through a frozen variant with `clf.classify_texts`. It uses the same rendering and cache. Outside this notebook, build the same `LLMClassifier` from `src/llm_classifier.py`.

In [ ]:
# Example:
# clf.classify_texts(["Your mailbox is almost full. Sign in within 24 hours to keep receiving email: http://..."])